# Installazione delle Librerie Necessarie

In [ ]:
# A. Gestiamo 'pandas' per compatibilità con 'google-colab'
print("Installazione/Verifica di pandas...")
!pip install pandas==2.2.2

# B. Risolviamo il conflitto fsspec / gcsfs
#    Installiamo la versione specifica di fsspec richiesta da gcsfs.
#    È meglio farlo PRIMA di installare altre librerie che potrebbero dipendere da fsspec.
print("\nInstallazione di fsspec per compatibilità con gcsfs...")
!pip install fsspec==2025.3.2

# C. Installiamo le librerie Hugging Face e altre dipendenze.
#    Come prima, NON includiamo 'torch' esplicitamente per usare la versione di Colab.
print("\nInstallazione di transformers, datasets, scikit-learn, accelerate...")
!pip install transformers datasets scikit-learn accelerate -U

print("\n--- Installazioni completate ---")
print("Per favore, controlla attentamente l'output qui sopra per eventuali messaggi di errore residui.")

# D. Verifica delle versioni chiave installate
print("\n--- Versioni Attuali delle Librerie Chiave ---")
import pandas
print(f"Pandas: {pandas.__version__} (attesa: 2.2.2)")
import torch
print(f"PyTorch: {torch.__version__} (attesa: una versione fornita da Colab, es. 2.6.0+cuXXX)")
import transformers
print(f"Transformers: {transformers.__version__}")
import datasets
print(f"Datasets: {datasets.__version__}")
import sklearn
print(f"Scikit-learn: {sklearn.__version__}")
import accelerate
print(f"Accelerate: {accelerate.__version__}")
import fsspec
print(f"fsspec: {fsspec.__version__} (attesa: 2025.3.2)")
import gcsfs
print(f"gcsfs: {gcsfs.__version__} (attesa: 2025.3.2 o compatibile con fsspec 2025.3.2)")


print("\nSe vedi ancora errori di dipendenza significativi, specialmente relativi a PyTorch,")
print("potrebbe essere necessario un altro riavvio del runtime e poi, PRIMA di installare")
print("transformers ecc., forzare una specifica versione di PyTorch, ad esempio:")
print("# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0")
print("(adattando le versioni a quelle richieste dalle librerie di Colab).")

Installazione/Verifica di pandas...

Installazione di fsspec per compatibilità con gcsfs...


# Importazioni Principali

In [3]:
# Importiamo tutte le librerie e i moduli necessari.
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback # Opzionale: per interrompere il training se non ci sono miglioramenti
)
import logging # Per un logging più dettagliato

# Impostazioni di logging (opzionale, ma utile per debug)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Librerie importate.")

Librerie importate.


# Montaggio di Google Drive e Definizione dei Percorsi

In [6]:
# Montaggio di Google Drive e Definizione dei Percorsi

from google.colab import drive
import os # os è già importato, ma per chiarezza

# 1. Monta Google Drive
# Verrà richiesto di autenticarsi e autorizzare l'accesso.
try:
    drive.mount('/content/drive', force_remount=True) # force_remount è utile se Drive era già montato
    print("Google Drive montato con successo in /content/drive")
except Exception as e:
    print(f"Errore durante il montaggio di Google Drive: {e}")
    print("Assicurati di aver autorizzato l'accesso a Google Drive.")
    # Potresti voler interrompere l'esecuzione qui se Drive è essenziale
    # raise SystemExit("Montaggio Google Drive fallito.")

# 2. Definisci il percorso base del tuo progetto su Google Drive


BASE_PROJECT_PATH = '/content/drive/MyDrive/CAMI/' # <--- MODIFICA QUESTO SE NECESSARIO

# 3. Definisci i percorsi specifici per dati, modelli e log
#    Questi saranno SOTTO la tua BASE_PROJECT_PATH
DATA_DIR_GDRIVE = os.path.join(BASE_PROJECT_PATH, 'data')
MODEL_OUTPUT_DIR_GDRIVE = os.path.join(BASE_PROJECT_PATH, 'cami_model_finetuned_colab')
LOGGING_DIR_GDRIVE = os.path.join(BASE_PROJECT_PATH, 'cami_logs_colab')
CSV_PATH_GDRIVE = os.path.join(DATA_DIR_GDRIVE, 'CAMI_dataset_v2.csv')

# 4. Crea le directory su Google Drive se non esistono
#    È buona pratica assicurarsi che esistano prima di provare a scrivere.
os.makedirs(BASE_PROJECT_PATH, exist_ok=True) # Crea la cartella base del progetto se non esiste
os.makedirs(DATA_DIR_GDRIVE, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR_GDRIVE, exist_ok=True)
os.makedirs(LOGGING_DIR_GDRIVE, exist_ok=True)

print(f"\nPercorso base del progetto su Drive: {BASE_PROJECT_PATH}")
print(f"Path del dataset su Drive: {CSV_PATH_GDRIVE}")
print(f"Directory di output del modello su Drive: {MODEL_OUTPUT_DIR_GDRIVE}")
print(f"Directory di logging su Drive: {LOGGING_DIR_GDRIVE}")

# 5. Verifica l'esistenza del file CSV (opzionale ma utile)
if os.path.exists(CSV_PATH_GDRIVE):
    print(f"\nFile CSV trovato in: {CSV_PATH_GDRIVE}")
else:
    print(f"\nATTENZIONE: File CSV NON TROVATO in {CSV_PATH_GDRIVE}")
    print("Assicurati che 'CAMI_dataset_v2.csv' sia presente nella sottocartella 'data' della tua cartella 'CAMI' su Google Drive.")
    print("Struttura attesa: /content/drive/MyDrive/CAMI/data/CAMI_dataset_v2.csv")
    # Qui potresti voler inserire la logica del dataset fittizio se il file non viene trovato,
    # ma è meglio assicurarsi che il file reale sia al posto giusto.

# Assicurati che le variabili globali usate più avanti puntino ai percorsi di Drive
CSV_PATH = CSV_PATH_GDRIVE
MODEL_OUTPUT_DIR = MODEL_OUTPUT_DIR_GDRIVE
LOGGING_DIR = LOGGING_DIR_GDRIVE

print(f"\nVariabili di percorso globali impostate per usare Google Drive:")
print(f"  CSV_PATH: {CSV_PATH}")
print(f"  MODEL_OUTPUT_DIR: {MODEL_OUTPUT_DIR}")
print(f"  LOGGING_DIR: {LOGGING_DIR}")

Mounted at /content/drive
Google Drive montato con successo in /content/drive

Percorso base del progetto su Drive: /content/drive/MyDrive/CAMI/
Path del dataset su Drive: /content/drive/MyDrive/CAMI/data/CAMI_dataset_v2.csv
Directory di output del modello su Drive: /content/drive/MyDrive/CAMI/cami_model_finetuned_colab
Directory di logging su Drive: /content/drive/MyDrive/CAMI/cami_logs_colab

File CSV trovato in: /content/drive/MyDrive/CAMI/data/CAMI_dataset_v2.csv

Variabili di percorso globali impostate per usare Google Drive:
  CSV_PATH: /content/drive/MyDrive/CAMI/data/CAMI_dataset_v2.csv
  MODEL_OUTPUT_DIR: /content/drive/MyDrive/CAMI/cami_model_finetuned_colab
  LOGGING_DIR: /content/drive/MyDrive/CAMI/cami_logs_colab


# Definizione Funzione per Caricamento e Preprocessing Dati

In [8]:
# Cella 5: Definizione Funzione per Caricamento e Preprocessing Dati

def load_and_preprocess_data_notebook(csv_path: str, test_size: float = 0.2, random_state: int = 42):
    logger = logging.getLogger(__name__) # Assicurati che logger sia definito
    try:
        df = pd.read_csv(csv_path, encoding='utf-8', sep=';') # AGGIUNTO sep=';'
        logger.info(f"Dataset caricato da {csv_path} con {len(df)} righe (usando ';' come separatore).")
        logger.info(f"Nomi delle colonne originali nel CSV (dopo sep=';'): {list(df.columns)}") # DEBUG
        logger.info(f"Prime 5 righe del CSV originale (dopo sep=';'):\n{df.head()}")
        logger.info(f"Dataset caricato da {csv_path} con {len(df)} righe.")
        logger.info(f"Nomi delle colonne originali nel CSV: {list(df.columns)}") # DEBUG CHIAVE
        logger.info(f"Prime 5 righe del CSV originale:\n{df.head()}") # DEBUG CHIAVE
    except FileNotFoundError:
        logger.error(f"File non trovato: {csv_path}")
        return None, None
    except Exception as e:
        logger.error(f"Errore durante il caricamento del CSV {csv_path}: {e}")
        return None, None

    NOME_COLONNA_TESTO_NEL_CSV = 'testo'
    NOME_COLONNA_ETICHETTA_NEL_CSV = 'etichetta'

    if NOME_COLONNA_TESTO_NEL_CSV not in df.columns:
        logger.error(f"COLONNA TESTO '{NOME_COLONNA_TESTO_NEL_CSV}' NON TROVATA nel CSV. Colonne disponibili: {list(df.columns)}")
        return None, None # Questo causa il SystemExit se la colonna non c'è
    if NOME_COLONNA_ETICHETTA_NEL_CSV not in df.columns:
        logger.error(f"COLONNA ETICHETTA '{NOME_COLONNA_ETICHETTA_NEL_CSV}' NON TROVATA nel CSV. Colonne disponibili: {list(df.columns)}")
        return None, None # Questo causa il SystemExit se la colonna non c'è

    df.rename(columns={
        NOME_COLONNA_TESTO_NEL_CSV: 'testo',
        NOME_COLONNA_ETICHETTA_NEL_CSV: 'etichetta'
    }, inplace=True)
    logger.info(f"Nomi colonne dopo il tentativo di ridenominazione in 'testo' e 'etichetta': {list(df.columns)}")

    # Da qui in poi, lo script si aspetta le colonne 'testo' e 'etichetta'.

    df.dropna(subset=['testo'], inplace=True)
    df.drop_duplicates(subset=['testo'], inplace=True)
    df['testo'] = df['testo'].astype(str).str.strip()
    df = df[df['testo'] != ""]
    logger.info(f"Numero di righe dopo pulizia 'testo': {len(df)}")

    # Gestione etichette: dato che hai detto che sono già 0/1 numerici,
    # la mappatura stringa è meno critica, ma la conversione e il controllo sono ancora importanti.
    if 'etichetta' not in df.columns: # Doppio controllo, non dovrebbe succedere se il rename ha funzionato
        logger.error("ERRORE INTERNO: La colonna 'etichetta' è sparita dopo la ridenominazione e la pulizia del testo.")
        return None, None

    logger.info(f"Valori unici in 'etichetta' (dopo rename, prima di to_numeric):\n{df['etichetta'].value_counts(dropna=False)}")

    # Tentativo di conversione a numerico. Se sono già int/float, non dovrebbe cambiare molto.
    # errors='coerce' trasformerà in NaN qualsiasi cosa non sia convertibile in numero.
    df['etichetta'] = pd.to_numeric(df['etichetta'], errors='coerce')
    logger.info(f"Valori unici in 'etichetta' DOPO to_numeric (con errors='coerce'):\n{df['etichetta'].value_counts(dropna=False)}")

    df.dropna(subset=['etichetta'], inplace=True) # Rimuove righe dove 'etichetta' è diventata NaN (es. se c'erano stringhe non numeriche)
    logger.info(f"Numero di righe dopo dropna su 'etichetta': {len(df)}")
    if df.empty:
        logger.error("Dataset vuoto dopo aver rimosso righe con etichette non valide (NaN). Controlla che la colonna etichetta contenga effettivamente solo 0 e 1 numerici e nessun altro valore/stringa.")
        return None, None

    # Assicura che siano interi e solo 0 o 1
    df['etichetta'] = df['etichetta'].astype(int) # Questo fallirà se ci sono float non interi (es. 0.5) o se ci sono NaN rimasti
    df = df[df['etichetta'].isin([0, 1])]
    logger.info(f"Numero di righe dopo aver filtrato per etichette [0, 1]: {len(df)}")
    if df.empty:
        logger.error("Dataset vuoto dopo aver filtrato per etichette [0, 1]. La colonna etichetta, anche se numerica, potrebbe non contenere solo 0 e 1.")
        return None, None

    if len(df) < 2:
        logger.error("Dataset troppo piccolo dopo la pulizia per essere suddiviso.")
        return None, None

    # ... resto della funzione (train_test_split, rename finale in 'label') ...
    if len(df['etichetta'].unique()) < 2:
        logger.warning("Attenzione: il dataset ha una sola classe dopo la pulizia. La stratificazione potrebbe fallire o non essere significativa.")
        train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state)
    else:
        try:
            train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df['etichetta'])
        except ValueError as e:
            logger.warning(f"Errore durante la stratificazione (es. una classe ha pochi campioni): {e}. Suddivisione non stratificata.")
            train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state)

    train_df = train_df.rename(columns={'etichetta': 'label'})
    val_df = val_df.rename(columns={'etichetta': 'label'})

    logger.info(f"Dataset suddiviso: {len(train_df)} righe per training, {len(val_df)} righe per validazione.")
    return train_df, val_df

# Assicurati che pandas e logging siano importati prima di definire la Cella 5
import pandas as pd
import logging
# Se non l'hai già fatto, configura il logger all'inizio del notebook (Cella 3 o simile)
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


print("Funzione 'load_and_preprocess_data_notebook' (aggiornata per etichette 0/1) definita.")

Funzione 'load_and_preprocess_data_notebook' (aggiornata per etichette 0/1) definita.


# Configurazioni del Modello e del Training

In [12]:
# Definiamo i parametri per il modello e il training.
MODEL_NAME = "Musixmatch/umberto-wikipedia-uncased-v1" # Modello UMBERTO base
# CSV_PATH è già definito nella cella 4

# Parametri di training
NUM_EPOCHS = 4                   # Numero di epoche di training
BATCH_SIZE_TRAIN = 8             # Batch size per il training (riduci se hai errori OOM - Out Of Memory)
BATCH_SIZE_EVAL = 16             # Batch size per la validazione
LEARNING_RATE = 2e-5             # Tasso di apprendimento
MAX_SEQ_LENGTH = 128             # Lunghezza massima delle sequenze tokenizzate
WEIGHT_DECAY = 0.01              # Decadimento del peso per regolarizzazione
RANDOM_SEED = 43                 # Seed per riproducibilità

print("Configurazioni caricate:")
print(f"  Modello base: {MODEL_NAME}")
print(f"  Numero epoche: {NUM_EPOCHS}")
print(f"  Batch size (train): {BATCH_SIZE_TRAIN}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")

Configurazioni caricate:
  Modello base: Musixmatch/umberto-wikipedia-uncased-v1
  Numero epoche: 5
  Batch size (train): 8
  Learning rate: 2e-05
  Max sequence length: 128


# Verifica e Impostazione del Dispositivo (GPU/CPU)

In [13]:
# Controlliamo se è disponibile una GPU e impostiamo il dispositivo di conseguenza.
# Questo è cruciale per velocizzare il training.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU disponibile: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Nessuna GPU trovata, il training verrà eseguito su CPU (potrebbe essere lento).")

GPU disponibile: Tesla T4


# Caricamento e Preparazione del Dataset

In [14]:
# Carichiamo i dati usando la funzione definita prima.
train_df, val_df = load_and_preprocess_data_notebook(CSV_PATH, test_size=0.2, random_state=RANDOM_SEED)

if train_df is not None and val_df is not None:
    print("\nPrime 5 righe del DataFrame di training:")
    print(train_df.head())
    print(f"\nNumero di campioni di training: {len(train_df)}")
    print(f"Numero di campioni di validazione: {len(val_df)}")

    # Convertiamo i DataFrame di Pandas in oggetti Dataset di Hugging Face
    train_dataset_hf = Dataset.from_pandas(train_df[['testo', 'label']])
    val_dataset_hf = Dataset.from_pandas(val_df[['testo', 'label']])

    print("\nDataset convertiti in formato Hugging Face:")
    print(train_dataset_hf)
    print(val_dataset_hf)
else:
    print("Errore nel caricamento dei dati. Controlla i log e il percorso del file CSV.")
    # Interrompi l'esecuzione se i dati non sono caricati
    raise SystemExit("Errore caricamento dati, training interrotto.")


Prime 5 righe del DataFrame di training:
                    Dataset di provenienza        tipo_metafora  \
1883  Dataset_Marchesetti_GenitiveConcrete             Genitive   
1668                  Dataset_MetaLiterary             Genitive   
826                    Dataset_Bambini2024    Nominal word pair   
234                         Dataset_MoveMe            Predicate   
290                    Dataset_Bambini2013  Nominal predicative   

                                           testo  label  argomento  \
1883                             deserto di neve      0    deserto   
1668                        girandola di sguardi      1  girandola   
826                             tenaglia bullone      0    bullone   
234   Il boscaiolo spacca il tronco con l' ascia      0     tronco   
290               Quegli uragani sono carrarmati      1    uragano   

         veicolo  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  Unnamed: 10  \
1883        neve         NaN         NaN         NaN 

# Caricamento Tokenizer e Modello Pre-addestrato

In [15]:
# Carichiamo il tokenizer e il modello UMBERTO.
# Il tokenizer converte il testo in input numerici che il modello può comprendere.
# Il modello è pre-addestrato su un vasto corpus e verrà fine-tunato sui nostri dati.

logger.info(f"Caricamento tokenizer per {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

logger.info(f"Caricamento modello {MODEL_NAME} per classificazione di sequenze...")
# num_labels=2 perché abbiamo due classi: metafora (1) e non-metafora (0).
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Spostiamo il modello sul dispositivo selezionato (GPU o CPU)
model.to(device)

print("Tokenizer e Modello caricati.")
if device.type == 'cuda':
    print(f"Modello spostato su {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/801k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.02M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Musixmatch/umberto-wikipedia-uncased-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer e Modello caricati.
Modello spostato su Tesla T4


# Definizione Funzione di Tokenizzazione e Applicazione

In [16]:
# Definiamo una funzione per tokenizzare i testi.
# 'padding="max_length"' assicura che tutte le sequenze abbiano la stessa lunghezza.
# 'truncation=True' taglia le sequenze più lunghe di MAX_SEQ_LENGTH.
def tokenize_function(examples):
    return tokenizer(
        examples['testo'],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

# Applichiamo la tokenizzazione ai nostri dataset.
# 'batched=True' processa più esempi contemporaneamente per efficienza.
logger.info("Tokenizzazione dei dataset in corso...")
train_tokenized_dataset = train_dataset_hf.map(tokenize_function, batched=True)
val_tokenized_dataset = val_dataset_hf.map(tokenize_function, batched=True)

# Rimuoviamo la colonna 'testo' originale perché non serve più al modello dopo la tokenizzazione.
# Rimuoviamo anche '__index_level_0__' se presente (artefatto di Dataset.from_pandas)
train_tokenized_dataset = train_tokenized_dataset.remove_columns(
    [col for col in ["testo", "__index_level_0__"] if col in train_tokenized_dataset.column_names]
)
val_tokenized_dataset = val_tokenized_dataset.remove_columns(
    [col for col in ["testo", "__index_level_0__"] if col in val_tokenized_dataset.column_names]
)


# Impostiamo il formato dei dataset su 'torch' per PyTorch.
train_tokenized_dataset.set_format("torch")
val_tokenized_dataset.set_format("torch")

print("\nDataset tokenizzati e formattati per PyTorch:")
print("Esempio dal dataset di training tokenizzato:")
print(train_tokenized_dataset[0])

Map:   0%|          | 0/1507 [00:00<?, ? examples/s]

Map:   0%|          | 0/377 [00:00<?, ? examples/s]


Dataset tokenizzati e formattati per PyTorch:
Esempio dal dataset di training tokenizzato:
{'label': tensor(0), 'input_ids': tensor([   5, 9336,   24, 9718,    6,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1]), 'attention_mask'

# Definizione Funzione per Calcolare le Metriche

In [17]:
# Definiamo una funzione per calcolare le metriche di valutazione (accuracy, F1, precision, recall).
# Questa funzione sarà chiamata dal Trainer durante la validazione.
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1) # Prendiamo l'indice della classe con la probabilità maggiore

    accuracy = accuracy_score(labels, preds)
    # 'binary' average è appropriato per classificazione binaria
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("Funzione 'compute_metrics' definita.")

Funzione 'compute_metrics' definita.


# Definizione degli Argomenti di Training

In [18]:
# Configuriamo gli argomenti per il training usando TrainingArguments.
# Questi argomenti controllano vari aspetti del processo di training.
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,          # Directory dove salvare il modello e i checkpoint
    num_train_epochs=NUM_EPOCHS,          # Numero totale di epoche di training
    per_device_train_batch_size=BATCH_SIZE_TRAIN, # Batch size per il training per dispositivo
    per_device_eval_batch_size=BATCH_SIZE_EVAL,   # Batch size per la validazione per dispositivo
    learning_rate=LEARNING_RATE,          # Tasso di apprendimento
    weight_decay=WEIGHT_DECAY,            # Forza della regolarizzazione weight decay

    eval_strategy="epoch",          # Strategia di valutazione (es. "steps", "epoch")
    save_strategy="epoch",                # Strategia di salvataggio dei checkpoint (es. "steps", "epoch")
                                          # Salvare ogni epoca aiuta a riprendere in caso di interruzioni

    logging_dir=LOGGING_DIR,              # Directory per i log (es. per TensorBoard)
    logging_strategy="epoch",             # Logga le metriche ad ogni epoca
    # logging_steps=10,                   # Alternativa: logga ogni N steps

    load_best_model_at_end=True,          # Carica il miglior modello (basato sulla metrica di validazione) alla fine del training
    metric_for_best_model="f1",           # Metrica per determinare il "miglior" modello (puoi scegliere 'loss', 'accuracy', etc.)
    greater_is_better=True,               # Per F1-score, un valore maggiore è migliore

    fp16=torch.cuda.is_available(),       # Usa precisione mista (float16) se GPU disponibile e supportata (accelera il training)
    # no_cuda=(device.type == 'cpu'),     # Decommenta se vuoi forzare l'uso della CPU

    report_to="tensorboard",              # Opzionale: integra con TensorBoard
    # report_to="none",                   # Disabilita reporting a servizi esterni

    seed=RANDOM_SEED,                     # Seed per la riproducibilità
    # dataloader_num_workers=2            # Se hai problemi di bottleneck I/O, puoi aumentarlo (es. in ambienti locali)
)

print("TrainingArguments configurati.")

TrainingArguments configurati.


# Creazione del Trainer

In [19]:
# Creiamo l'oggetto Trainer, che gestirà il ciclo di training e valutazione.
trainer = Trainer(
    model=model,                          # Il modello da fine-tunare
    args=training_args,                   # Gli argomenti di training definiti sopra
    train_dataset=train_tokenized_dataset,# Il dataset di training tokenizzato
    eval_dataset=val_tokenized_dataset,   # Il dataset di validazione tokenizzato
    tokenizer=tokenizer,                  # Il tokenizer (utile per padding dinamico e salvataggio)
    compute_metrics=compute_metrics,      # La funzione per calcolare le metriche
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Opzionale: aggiungi callback per early stopping
)

print("Trainer creato.")

Trainer creato.


<ipython-input-19-8f9b29696f1d>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Avvio del Training

In [20]:
# Avviamo il processo di training!
# Il Trainer mostrerà l'andamento della loss e delle metriche per ogni epoca.
logger.info("Inizio del training...")

try:
    train_result = trainer.train()
    logger.info("Training completato con successo.")

    # Stampiamo un riepilogo delle metriche di training
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)

except Exception as e:
    logger.error(f"Errore durante il training: {e}")
    # Potrebbe essere utile salvare lo stato attuale se il training si interrompe
    # trainer.save_model(os.path.join(MODEL_OUTPUT_DIR, "interrupted_checkpoint"))
    # logger.info("Checkpoint del modello interrotto salvato.")
    raise # Rilancia l'eccezione per vedere il traceback completo

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.647400,0.626310,0.657825,0.724947,0.739130,0.711297
2,0.611000,0.582277,0.687003,0.752101,0.755274,0.748954
3,0.565200,0.550659,0.737401,0.795876,0.784553,0.807531
4,0.513200,0.534111,0.748011,0.801670,0.800000,0.803347
5,0.442100,0.570925,0.742706,0.790497,0.816964,0.765690


***** train metrics *****
  epoch                    =        5.0
  total_flos               =   461596GF
  train_loss               =     0.5558
  train_runtime            = 0:05:08.94
  train_samples_per_second =      24.39
  train_steps_per_second   =      3.059


# Salvataggio del Modello Fine-tunato

In [20]:
# Dopo il training, salviamo il modello fine-tunato e il tokenizer.
# Se `load_best_model_at_end=True`, il trainer avrà già caricato il miglior modello.
logger.info(f"Salvataggio del modello fine-tunato in {MODEL_OUTPUT_DIR}...")
trainer.save_model(MODEL_OUTPUT_DIR) # Salva il modello
# Il tokenizer viene salvato automaticamente con save_model se fornito al Trainer,
# ma è buona pratica salvarlo esplicitamente per chiarezza o se si vuole un path diverso.
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"Modello e tokenizer fine-tunati salvati in: {MODEL_OUTPUT_DIR}")
print("\nContenuto della directory del modello:")
print(os.listdir(MODEL_OUTPUT_DIR))

Modello e tokenizer fine-tunati salvati in: /content/drive/MyDrive/CAMI/cami_model_finetuned_colab

Contenuto della directory del modello:
['checkpoint-189', 'checkpoint-378', 'checkpoint-567', 'train_results.json', 'all_results.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'sentencepiece.bpe.model', 'tokenizer.json', 'training_args.bin']


# Test Rapido del Modello Caricato (Inferenza Semplice)

In [11]:
logger.info("Test rapido del modello caricato...")

# Carica il modello e il tokenizer salvati
loaded_tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(MODEL_OUTPUT_DIR)
loaded_model.to(device) # Sposta sulla GPU/CPU
loaded_model.eval()     # Metti in modalità valutazione

# Frasi di esempio per il test
test_sentences = [
    "Il mare è una tavola",
    "questi poliziotti sono violenti",
    "Chiara è bravissima"
]

print("\nPredizioni su frasi di esempio:")
for sentence in test_sentences:
    inputs = loaded_tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Sposta i dati sul device

    with torch.no_grad():
        outputs = loaded_model(**inputs)
        logits = outputs.logits
        prediction = torch.argmax(logits, dim=-1).item() # 0 o 1

    label_map = {0: "Non-Metafora", 1: "Metafora"}
    print(f"Frase: '{sentence}' -> Predizione: {label_map[prediction]} ({prediction})")

logger.info("Test rapido completato.")


Predizioni su frasi di esempio:
Frase: 'Il mare è una tavola' -> Predizione: Metafora (1)
Frase: 'questi poliziotti sono violenti' -> Predizione: Metafora (1)
Frase: 'Chiara è bravissima' -> Predizione: Metafora (1)
